In [62]:
# ==============================================================================
# 0. CONFIGURATION
# ==============================================================================

# ── Paths ──────────────────────────────────────────────────────────────────────

FULL_OUTPUT_DIR = "Results/Regular_clustering/Full_dataset"

# ── Paths ──────────────────────────────────────────────────────────────────────

# 5 clusters -------------------------------------------------------
CLUSTER_LABELS_5 = {
     1:  "C1 — UHCD + Hospitalization + heavy workup",
     0:  "C2 — Hospitalized + full workup",
     2:  "C3 — Discharged + biology +/- ECG",
     4:  "C4 — Isolated X-ray +/- CT",
     3:  "C5 — Minimal consumption",
     -1: "Outliers",
 }

# Ordre logique du plus au moins consommateur
CLUSTER_ORDER_5 = ["C1 — UHCD + Hospitalization + heavy workup",
                  "C2 — Hospitalized + full workup",
                  "C3 — Discharged + biology +/- ECG",
                  "C4 — Isolated X-ray +/- CT",
                  "C5 — Minimal consumption",
                  "Outliers"]


# 9 clusters -------------------------------------------------------
CLUSTER_LABELS_9 = {
    0:  "C1 — UHCD + Hospitalization + heavy workup",   # obs=1, hospi=1, bio=1.56, ekg=0.55
    4:  "C2 — Hospitalized + biology + imaging",         # hospi=1, bio=2.19, ct=0.74, ekg=0.34
    5:  "C3 — Hospitalized + biology",                   # hospi=1, bio=0.40, blood=0.20
    6:  "C4 — Hospitalized + ECG + CT",                  # hospi=1, ct=1.00, ekg=0.30, bio=1.00
    7:  "C5 — Hospitalized + ultrasound + biology",      # hospi=1, echo=0.98, bio=1.00
    8:  "C6 — Hospitalized + ECG",                       # hospi=1, ekg=0.48, bio=1.00
    1:  "C7 — Discharged + biology",                     # hospi=0, blood=1.00, bio=1.00
    3:  "C8 — Isolated X-ray",                           # xray=1.00, imaging=1.11
    2:  "C9 — Minimal consumption",                      # tout à 0
    -1: "Outliers",
}

CLUSTER_ORDER_9 = [
    "C1 — UHCD + Hospitalization + heavy workup",
    "C2 — Hospitalized + biology + imaging",
    "C3 — Hospitalized + biology",
    "C4 — Hospitalized + ECG + CT",
    "C5 — Hospitalized + ultrasound + biology",
    "C6 — Hospitalized + ECG",
    "C7 — Discharged + biology",
    "C8 — Isolated X-ray",
    "C9 — Minimal consumption",
    "Outliers",
]

# ── Description features ───────────────────────────────────────────────────
MEASURED_STATUS_PAIRS = {
    "is_bp_measured":                     "bp_status",
    "is_ht_measured":                     "ht_status",
    "is_temp_measured":                   "temp_status",
    "is_sat_measured":                    "sat_status",
    "is_rr_measured":                     "rr_status",
    "is_o2_measured":                     "o2_flow_status",
    "is_gcs_measured":                    "gcs_status",
    "is_cap_blood_sugar_mmol_L_measured": "cap_blood_sugar_status",
    "is_pupil_right_measured":            "anisocoria_status",
    "is_urine_dipstick_clean_measured":   "urine_dipstick_clean_status",
    "is_pain_measured":                   "pain_status",
    "is_breathalyzer_measured":           "breathalyzer_status",
    "is_hemocue_measured":                "hemocue_status",
}

ADMISSION_QUANTI = [
    "age",
    #"duration_triage_ioa_min",
    "triage"
]

ADMISSION_CATEG = [
    "age_group",
    "sex",
    "transport_grouped",

]

CHIEF_COMPLAINT_COL = "chief_complaint"
TOP_N_COMPLAINTS    = 10


COMPLAINT_CAT_COL = "complaint_category"
N_CATEGORY = 17
# ==============================================================================
# 1. IMPORTS
# ==============================================================================

import os
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

In [63]:
# ==============================================================================
# 1. CLUSTERS DESCRIPTION — EXTERNAL FEATURES (ADMISSION, TRIAGE, CHIEF COMPLAINT, IS_MEASURED, STATUS)
# ==============================================================================
# labelling clusters
def remap_cluster_labels(labels: np.ndarray, cluster_labels: dict) -> np.ndarray:
    """Remaps integer cluster labels to descriptive string labels."""
    return np.array([cluster_labels.get(l, f"Unknown ({l})") for l in labels])

# ==============================================================================
# 1. MAIN FUNCTION
# ==============================================================================

def describe_clusters(
    df:           pd.DataFrame,
    labels:       np.ndarray,
    idx:          pd.Index,
    run_label:    str,
    out_dir:      str,
    df_sub:       pd.DataFrame = None,
    cluster_labels: dict = None,
    cluster_order:  list = None,
):
    """
    Generate a complete description of clusters :
    1. Clustering variables (heatmap already generated, here stats tables)
    2. Admission and triage variables

    Parameters
    ----------
    df        : Full DataFrame (all variables)
    labels    : HDBSCAN label array
    idx       : Index of patients used in clustering
    run_label : Run identifier
    out_dir   : Output directory
    df_sub    : DataFrame of clustering variables (optional)
    """
    os.makedirs(out_dir, exist_ok=True)

    # Rebuilding a df with labels and external variables for description
    df_desc           = df.loc[idx].copy()
    df_desc["cluster"] = labels
    df_desc           = df_desc[df_desc["cluster"] != -1]  # exclure le bruit

    # ── Remapping ─────────────────────────────────────────────────────────────
    if cluster_labels:
        df_desc["cluster"] = df_desc["cluster"].map(cluster_labels)
        clusters = cluster_order if cluster_order else sorted(df_desc["cluster"].unique())
    else:
        clusters = sorted(df_desc["cluster"].unique())

    n_total    = len(df_desc)
    n_clusters = len(clusters)
    log.info(f"[{run_label}] Description of {n_clusters} clusters ({n_total} patients)")



    # ── 1. Clusters size ─────────────────────────────────────────────────
    _plot_cluster_sizes(df_desc, clusters, run_label, out_dir)

    # ── 2. Quantitative variables from triage ───────────────────────────────────
    quanti_available = [c for c in ADMISSION_QUANTI if c in df_desc.columns]
    if quanti_available:
        _plot_quanti_by_cluster(df_desc, quanti_available, clusters, run_label, out_dir)

    # ── 3. Categorical variables from triage ──────────────────────────────────
    categ_available = [c for c in ADMISSION_CATEG if c in df_desc.columns]
    if categ_available:
        _plot_categ_by_cluster(df_desc, categ_available, clusters, run_label, out_dir)

    # ── 4. Triage (ordinal) ───────────────────────────────────────────────────
    if "triage" in df_desc.columns:
        _plot_triage_by_cluster(df_desc, clusters, run_label, out_dir)

    # ── 5. Chief complaint — top 10 per cluster ───────────────────────────────
    if CHIEF_COMPLAINT_COL in df_desc.columns:
        _plot_chief_complaint(df_desc, clusters, run_label, out_dir)

    # ── 5b. Chief complaint category  ───────────────────────────────
    if COMPLAINT_CAT_COL in df_desc.columns:
        _plot_complaint_categories(df_desc, clusters, run_label, out_dir)

    # ── 6. Variables is_measured ──────────────────────────────────────────────
    _plot_measured_vars(df_desc, clusters, run_label, out_dir)

    # ── 7. Reverse distribution ───────────────────────────────────────────────
    _plot_reverse_distribution(
        df_desc, clusters, run_label,
        out_dir=os.path.join(out_dir, "reverse_distribution")
    )

    # ── 8. Export summary table CSV ───────────────────────────────────
    _export_summary_table(df_desc, clusters, quanti_available, categ_available, run_label, out_dir)

    log.info(f"[{run_label}] Description done → {out_dir}")


# ==============================================================================
# 2. SUB FUNCTIONS FOR CLUSTER DESCRIPTION
# ==============================================================================

def _plot_cluster_sizes(df_desc, clusters, run_label, out_dir):
    sizes = df_desc["cluster"].value_counts().sort_index()
    pcts  = (sizes / sizes.sum() * 100).round(1)

    fig, ax = plt.subplots(figsize=(max(6, len(clusters) * 1.2), 5))
    bars = ax.bar(
        [f"C{c}" for c in sizes.index],
        sizes.values,
        color=sns.color_palette("tab10", len(clusters))
    )
    for bar, pct in zip(bars, pcts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + sizes.max() * 0.01,
            f"{pct}%", ha="center", va="bottom", fontsize=10
        )
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Number of patients")
    ax.set_title(f"Clusters size — {run_label}")
    plt.tight_layout()
    path = os.path.join(out_dir, "cluster_sizes.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")


def _plot_quanti_by_cluster(df_desc, quanti_cols, clusters, run_label, out_dir):
    n     = len(quanti_cols)
    fig, axes = plt.subplots(1, n, figsize=(n * 5, 5))
    if n == 1:
        axes = [axes]

    palette = sns.color_palette("tab10", len(clusters))

    for ax, col in zip(axes, quanti_cols):
        data = [df_desc[df_desc["cluster"] == c][col].dropna().values for c in clusters]
        bp = ax.boxplot(data, patch_artist=True, labels=[f"C{c}" for c in clusters])
        for patch, color in zip(bp["boxes"], palette):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_title(col)
        ax.set_xlabel("Cluster")

    plt.suptitle(f"Quantitatives variables — {run_label}", y=1.02)
    plt.tight_layout()
    path = os.path.join(out_dir, "admission_quanti.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")


def _plot_categ_by_cluster(df_desc, categ_cols, clusters, run_label, out_dir):
    for col in categ_cols:
        if col not in df_desc.columns:
            continue

        # Proportion de chaque modalité par cluster
        ct = (
            df_desc.groupby(["cluster", col])
            .size()
            .reset_index(name="n")
        )
        ct["pct"] = ct.groupby("cluster")["n"].transform(lambda x: x / x.sum() * 100)

        modalities = df_desc[col].dropna().unique()
        n_mod      = len(modalities)

        fig, ax = plt.subplots(figsize=(max(8, len(clusters) * 1.5), 5))
        x       = np.arange(len(clusters))
        width   = 0.8 / n_mod
        palette = sns.color_palette("tab10", n_mod)

        for i, mod in enumerate(sorted(modalities)):
            vals = []
            for c in clusters:
                row = ct[(ct["cluster"] == c) & (ct[col] == mod)]
                vals.append(row["pct"].values[0] if len(row) > 0 else 0)
            ax.bar(x + i * width, vals, width, label=str(mod), color=palette[i], alpha=0.8)

        ax.set_xticks(x + width * (n_mod - 1) / 2)
        ax.set_xticklabels([f"C{c}" for c in clusters])
        ax.set_ylabel("% patients")
        ax.set_title(f"{col} par cluster — {run_label}")
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        path = os.path.join(out_dir, f"admission_{col}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")


def _plot_triage_by_cluster(df_desc, clusters, run_label, out_dir):
    triage_levels = sorted(df_desc["triage"].dropna().unique())
    ct = (
        df_desc.groupby(["cluster", "triage"])
        .size()
        .reset_index(name="n")
    )
    ct["pct"] = ct.groupby("cluster")["n"].transform(lambda x: x / x.sum() * 100)

    n_levels = len(triage_levels)
    x        = np.arange(len(clusters))
    width    = 0.8 / n_levels
    palette  = sns.color_palette("RdYlGn_r", n_levels)

    fig, ax = plt.subplots(figsize=(max(8, len(clusters) * 1.5), 5))
    for i, level in enumerate(triage_levels):
        vals = []
        for c in clusters:
            row = ct[(ct["cluster"] == c) & (ct["triage"] == level)]
            vals.append(row["pct"].values[0] if len(row) > 0 else 0)
        ax.bar(x + i * width, vals, width,
               label=f"Triage {int(level)}", color=palette[i], alpha=0.85)

    ax.set_xticks(x + width * (n_levels - 1) / 2)
    ax.set_xticklabels([f"C{c}" for c in clusters])
    ax.set_ylabel("% patients")
    ax.set_title(f"Triage per cluster — {run_label}")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    path = os.path.join(out_dir, "admission_triage.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")


def _plot_chief_complaint(df_desc, clusters, run_label, out_dir):
    # Top 10 global
    top10 = (
        df_desc[CHIEF_COMPLAINT_COL]
        .value_counts()
        .head(TOP_N_COMPLAINTS)
        .index.tolist()
    )

    ct = (
        df_desc[df_desc[CHIEF_COMPLAINT_COL].isin(top10)]
        .groupby(["cluster", CHIEF_COMPLAINT_COL])
        .size()
        .reset_index(name="n")
    )
    ct["pct"] = ct.groupby("cluster")["n"].transform(lambda x: x / x.sum() * 100)

    # Un graphique par cluster
    n_cols_fig = min(3, len(clusters))
    n_rows_fig = (len(clusters) + n_cols_fig - 1) // n_cols_fig
    fig, axes  = plt.subplots(n_rows_fig, n_cols_fig,
                               figsize=(n_cols_fig * 6, n_rows_fig * 5))
    axes = np.array(axes).flatten()

    palette = sns.color_palette("tab10", TOP_N_COMPLAINTS)

    for i, c in enumerate(clusters):
        ax   = axes[i]
        data = ct[ct["cluster"] == c].sort_values("pct", ascending=True)
        ax.barh(data[CHIEF_COMPLAINT_COL], data["pct"],
                color=palette[:len(data)], alpha=0.8)
        ax.set_title(f"Cluster {c}")
        ax.set_xlabel("% patients")

    # Cacher les axes vides
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(f"Top {TOP_N_COMPLAINTS} chief complaints — {run_label}", y=1.02)
    plt.tight_layout()
    path = os.path.join(out_dir, "chief_complaint_by_cluster.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")




def _plot_complaint_categories(df_desc, clusters, run_label, out_dir):

    all_categories = sorted(df_desc[COMPLAINT_CAT_COL].dropna().unique())

    ct = (
        df_desc.groupby(["cluster", COMPLAINT_CAT_COL])
        .size()
        .reset_index(name="n")
    )
    ct["pct"] = ct.groupby("cluster")["n"].transform(lambda x: x / x.sum() * 100)

    # One subplot per cluster
    n_cols_fig = min(3, len(clusters))
    n_rows_fig = (len(clusters) + n_cols_fig - 1) // n_cols_fig
    fig, axes  = plt.subplots(n_rows_fig, n_cols_fig,
                               figsize=(n_cols_fig * 7, n_rows_fig * 6))
    axes = np.array(axes).flatten()

    # Fixed palette — one color per category, consistent across subplots
    palette    = sns.color_palette("tab20", len(all_categories))
    color_map  = {cat: palette[i] for i, cat in enumerate(all_categories)}

    for i, c in enumerate(clusters):
        ax   = axes[i]
        data = ct[ct["cluster"] == c].sort_values("pct", ascending=True)
        colors = [color_map[cat] for cat in data[COMPLAINT_CAT_COL]]
        ax.barh(data[COMPLAINT_CAT_COL], data["pct"],
                color=colors, alpha=0.85)
        ax.set_title(f"Cluster {c}", fontsize=11)
        ax.set_xlabel("% patients")

    # Hide empty axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(f"Complaint categories by cluster — {run_label}", y=1.02)
    plt.tight_layout()
    path = os.path.join(out_dir, "complaint_categories_by_cluster.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")


def _plot_measured_vars(df_desc, clusters, run_label, out_dir):
    available_pairs = {
        k: v for k, v in MEASURED_STATUS_PAIRS.items()
        if k in df_desc.columns
    }
    if not available_pairs:
        return

    # ── Measure rate per cluster ─────────────────────────────────────────────
    measured_rates = pd.DataFrame(index=[f"C{c}" for c in clusters])
    for is_col in available_pairs:
        rates = []
        for c in clusters:
            sub  = df_desc[df_desc["cluster"] == c][is_col]
            rates.append(sub.mean() * 100 if len(sub) > 0 else 0)
        measured_rates[is_col.replace("is_", "").replace("_measured", "")] = rates

    fig, ax = plt.subplots(figsize=(max(10, len(available_pairs) * 1.2), 5))
    sns.heatmap(
        measured_rates.T, annot=True, fmt=".1f", cmap="YlOrRd",
        ax=ax, annot_kws={"size": 9}
    )
    ax.set_title(f"Taux de mesure (%) par cluster — {run_label}")
    ax.set_xlabel("Cluster")
    plt.tight_layout()
    path = os.path.join(out_dir, "measured_rates.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")

    # ── Status per variable (only measured patients) ─────────────────────
    for is_col, status_col in available_pairs.items():
        if status_col not in df_desc.columns:
            continue

        var_name = is_col.replace("is_", "").replace("_measured", "")

        # Filter only measured patients
        df_measured = df_desc[df_desc[is_col] == 1].copy()
        if len(df_measured) == 0:
            continue

        modalities = df_measured[status_col].dropna().unique()
        if len(modalities) == 0:
            continue

        ct = (
            df_measured.groupby(["cluster", status_col])
            .size()
            .reset_index(name="n")
        )
        ct["pct"] = ct.groupby("cluster")["n"].transform(
            lambda x: x / x.sum() * 100
        )

        n_mod   = len(modalities)
        x       = np.arange(len(clusters))
        width   = 0.8 / n_mod
        palette = sns.color_palette("tab10", n_mod)

        fig, ax = plt.subplots(figsize=(max(8, len(clusters) * 1.5), 5))
        for i, mod in enumerate(sorted(modalities)):
            vals = []
            for c in clusters:
                row = ct[(ct["cluster"] == c) & (ct[status_col] == mod)]
                vals.append(row["pct"].values[0] if len(row) > 0 else 0)
            ax.bar(x + i * width, vals, width,
                   label=str(mod), color=palette[i], alpha=0.8)

        ax.set_xticks(x + width * (n_mod - 1) / 2)
        ax.set_xticklabels([f"C{c}" for c in clusters])
        ax.set_ylabel("% Measured patients")
        ax.set_title(f"{var_name} status (Measured patients) — {run_label}")
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        path = os.path.join(out_dir, f"status_{var_name}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

def _plot_reverse_distribution(df_desc, clusters, run_label, out_dir):
    """
    Reverse reading: for each modality of each variable,
    how are patients distributed across clusters?
    e.g. 'Among triage 1 patients: 15% cluster 0, 40% cluster 2...'
    """
    os.makedirs(out_dir, exist_ok=True)
    palette = sns.color_palette("tab10", len(clusters))
    summary_rows = []

    all_vars = {}

    # Triage
    if "triage" in df_desc.columns:
        all_vars["triage"] = sorted(df_desc["triage"].dropna().unique())

    # Categorical
    for col in ADMISSION_CATEG:
        if col in df_desc.columns:
            all_vars[col] = sorted(df_desc[col].dropna().unique())

    # is_measured (0/1)
    for is_col in MEASURED_STATUS_PAIRS:
        if is_col in df_desc.columns:
            var_name = is_col.replace("is_", "").replace("_measured", "")
            all_vars[is_col] = [0, 1]

    # Status
    for status_col in MEASURED_STATUS_PAIRS.values():
        if status_col in df_desc.columns:
            all_vars[status_col] = sorted(df_desc[status_col].dropna().unique())

    for var, modalities in all_vars.items():
        n_mod = len(modalities)
        if n_mod == 0:
            continue

        n_cols_fig = min(3, n_mod)
        n_rows_fig = (n_mod + n_cols_fig - 1) // n_cols_fig
        fig, axes  = plt.subplots(
            n_rows_fig, n_cols_fig,
            figsize=(n_cols_fig * 5, n_rows_fig * 4)
        )
        axes = np.array(axes).flatten()

        for i, mod in enumerate(modalities):
            ax      = axes[i]
            df_mod  = df_desc[df_desc[var] == mod]
            n_mod_total = len(df_mod)

            if n_mod_total == 0:
                ax.set_visible(False)
                continue

            pcts = []
            for c in clusters:
                n_c = (df_mod["cluster"] == c).sum()
                pcts.append(round(100 * n_c / n_mod_total, 1))

            bars = ax.bar(
                [f"C{c}" for c in clusters],
                pcts,
                color=palette,
                alpha=0.85
            )
            for bar, pct in zip(bars, pcts):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.5,
                    f"{pct}%", ha="center", va="bottom", fontsize=8
                )

            ax.set_title(f"{var} = {mod}\n(n={n_mod_total:,})")
            ax.set_ylabel("% of patients")
            ax.set_xlabel("Cluster")
            ax.set_ylim(0, max(pcts) * 1.2 + 5)

            # Summary row
            row = {"variable": var, "modality": str(mod), "n": n_mod_total}
            for c, pct in zip(clusters, pcts):
                row[f"cluster_{c}_pct"] = pct
            summary_rows.append(row)

        # Hide empty axes
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)

        var_clean = var.replace("is_", "").replace("_measured", "")
        plt.suptitle(
            f"Cluster distribution by {var_clean} — {run_label}",
            y=1.02
        )
        plt.tight_layout()
        path = os.path.join(out_dir, f"reverse_{var_clean}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

    # Export summary CSV
    df_summary = pd.DataFrame(summary_rows)
    path = os.path.join(out_dir, "reverse_distribution_summary.csv")
    df_summary.to_csv(path, index=False)
    log.info(f"Saved: {path}")

def _export_summary_table(df_desc, clusters, quanti_cols, categ_cols, run_label, out_dir):
    """summary table CSV — mean/median quanti + % categ per cluster."""
    rows = []

    for c in clusters:
        sub  = df_desc[df_desc["cluster"] == c]
        row  = {"cluster": c, "n": len(sub), "pct_total": round(len(sub) / len(df_desc) * 100, 1)}

        # Quanti
        for col in quanti_cols:
            if col in sub.columns:
                row[f"{col}_mean"]   = round(sub[col].mean(), 1)
                row[f"{col}_median"] = round(sub[col].median(), 1)
                row[f"{col}_std"]    = round(sub[col].std(), 1)

        # Categ — dominant modality + % of this modality
        for col in categ_cols:
            if col in sub.columns:
                mode = sub[col].mode()
                row[f"{col}_mode"] = mode.iloc[0] if len(mode) > 0 else None
                row[f"{col}_mode_pct"] = round(
                    (sub[col] == row[f"{col}_mode"]).mean() * 100, 1
                ) if row[f"{col}_mode"] else None

        # Triage — distribution
        if "triage" in sub.columns:
            for level in sorted(df_desc["triage"].dropna().unique()):
                row[f"triage_{int(level)}_pct"] = round(
                    (sub["triage"] == level).mean() * 100, 1
                )

        # is_measured — Rate
        for is_col in MEASURED_STATUS_PAIRS:
            if is_col in sub.columns:
                var_name = is_col.replace("is_", "").replace("_measured", "")
                row[f"{var_name}_measured_pct"] = round(sub[is_col].mean() * 100, 1)

        rows.append(row)

    summary = pd.DataFrame(rows)
    path    = os.path.join(out_dir, "cluster_summary.csv")
    summary.to_csv(path, index=False)
    log.info(f"Saved: {path}")
    return summary

In [64]:
# ==============================================================================
# 2. OUTLIERS DESCRIPTION — EXTERNAL FEATURES (ADMISSION, TRIAGE, CHIEF COMPLAINT, IS_MEASURED, STATUS)
# ==============================================================================


def describe_outliers_external(
    df_clust:  pd.DataFrame,
    labels:    np.ndarray,
    idx:       pd.Index,
    run_label: str,
    out_dir:   str,
    cluster_labels: dict = None,
    cluster_order:  list = None,
):
    """
    Description of outliers using external variables
    (admission, triage, chief complaint, is_measured, status variables).
    To be called from the description notebook.
    """
    os.makedirs(out_dir, exist_ok=True)

    df_work            = df_clust.loc[idx].copy()
    df_work["cluster"] = labels

    if cluster_labels:
        df_work["cluster"] = df_work["cluster"].map(
            {**cluster_labels, -1: "Outliers"}
        )
        unique_clusters = cluster_order if cluster_order else [
            c for c in df_work["cluster"].unique() if c != "Outliers"
        ]
    else:
        unique_clusters = sorted([c for c in df_work["cluster"].unique() if c != -1])


    df_noise     = df_work[df_work["cluster"] == -1]
    df_clustered = df_work[df_work["cluster"] != -1]

    n_noise = len(df_noise)
    n_total = len(df_work)

    log.info(
        f"[{run_label}] Outliers : {n_noise} / {n_total} "
        f"({100*n_noise/n_total:.1f}%)"
    )

    if n_noise == 0:
        log.info("No outliers found.")
        return


    # ── Triage ─────────────────────────────────────────────────────────────────────
    if "triage" in df_noise.columns:

        # ── Graphique 1 — composition des outliers par triage ─────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        triage_dist = df_noise["triage"].value_counts().sort_index()
        triage_pct  = (triage_dist / triage_dist.sum() * 100).round(1)
        palette     = sns.color_palette("RdYlGn_r", len(triage_dist))

        axes[0].bar(
            [f"Triage {int(t)}" for t in triage_dist.index],
            triage_pct.values, color=palette, alpha=0.85
        )
        axes[0].set_ylabel("% outliers")
        axes[0].set_title("Triage distribution — outliers")

        rates = []
        for level in sorted(df_work["triage"].dropna().unique()):
            n_lev       = (df_work["triage"] == level).sum()
            n_noise_lev = (df_noise["triage"] == level).sum()
            rates.append({
                "triage":      int(level),
                "pct_outlier": round(100 * n_noise_lev / n_lev, 1) if n_lev > 0 else 0
            })
        df_rates = pd.DataFrame(rates)
        axes[1].bar(
            [f"Triage {t}" for t in df_rates["triage"]],
            df_rates["pct_outlier"],
            color=sns.color_palette("RdYlGn_r", len(df_rates)), alpha=0.85
        )
        axes[1].set_ylabel("% classified as noise")
        axes[1].set_title("Outlier rate by triage level")

        plt.suptitle(f"Triage — outliers — {run_label}")
        plt.tight_layout()
        path = os.path.join(out_dir, "outliers_triage.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

        # ── Graphique 2 — distribution cluster+outlier par niveau de triage ───────
        triage_levels   = sorted(df_work["triage"].dropna().unique())
        unique_clusters = sorted([c for c in df_work["cluster"].unique() if c != -1])

        # Construire un df : pour chaque niveau de triage →
        # % dans chaque cluster + % outliers
        rows_triage = []
        for level in triage_levels:
            df_level = df_work[df_work["triage"] == level]
            n_level  = len(df_level)
            if n_level == 0:
                continue
            row = {"triage": int(level), "n_total": n_level}
            for c in unique_clusters:
                row[f"cluster_{c}"] = round(
                    (df_level["cluster"] == c).sum() / n_level * 100, 1
                )
            row["outliers"] = round(
                (df_level["cluster"] == -1).sum() / n_level * 100, 1
            )
            rows_triage.append(row)

        df_triage_dist = pd.DataFrame(rows_triage).set_index("triage")

        # Stacked bar — un bar par niveau de triage
        cluster_cols_plot = [f"cluster_{c}" for c in unique_clusters]
        all_cols_plot     = cluster_cols_plot + ["outliers"]

        # Palette — clusters en tab20, outliers en gris
        n_clusters      = len(unique_clusters)
        cluster_palette = sns.color_palette("tab20", n_clusters) \
                          if n_clusters <= 20 \
                          else sns.color_palette("hsv", n_clusters)
        color_list      = list(cluster_palette) + ["lightgrey"]

        fig, ax = plt.subplots(figsize=(max(8, len(triage_levels) * 1.5), 6))
        bottom  = np.zeros(len(df_triage_dist))

        for col, color in zip(all_cols_plot, color_list):
            values = df_triage_dist[col].values
            ax.bar(
                [f"Triage {t}" for t in df_triage_dist.index],
                values, bottom=bottom,
                label=col.replace("cluster_", "C").replace("outliers", "Outliers"),
                color=color, alpha=0.85, edgecolor="white",
            )
            bottom += values

        ax.set_ylabel("% patients")
        ax.set_xlabel("Triage level")
        ax.set_title(
            f"Cluster + outlier distribution by triage level — {run_label}"
        )
        ax.legend(
            title="Cluster", bbox_to_anchor=(1.05, 1),
            loc="upper left", fontsize=8,
        )
        ax.set_ylim(0, 105)

        # Annoter le % outliers sur chaque barre
        for i, (level, row) in enumerate(df_triage_dist.iterrows()):
            pct_out = row["outliers"]
            n_tot   = row["n_total"]
            ax.text(
                i, 102,
                f"n={int(n_tot)}\n{pct_out}% noise",
                ha="center", va="bottom", fontsize=8,
            )

        plt.tight_layout()
        path = os.path.join(out_dir, "triage_cluster_distribution.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

        # Export CSV
        df_triage_dist["n_total"] = [
            (df_work["triage"] == t).sum() for t in df_triage_dist.index
        ]
        df_triage_dist.to_csv(
            os.path.join(out_dir, "triage_cluster_distribution.csv")
        )
        log.info(f"Saved: triage_cluster_distribution.csv")

    # ── Quantitative admission variables ───────────────────────────────────────
    quanti_available = [c for c in ADMISSION_QUANTI if c in df_noise.columns]
    if quanti_available:
        fig, axes = plt.subplots(1, len(quanti_available),
                                  figsize=(len(quanti_available) * 5, 5))
        if len(quanti_available) == 1:
            axes = [axes]
        for ax, col in zip(axes, quanti_available):
            ax.boxplot(
                [df_noise[col].dropna().values,
                 df_clustered[col].dropna().values],
                patch_artist=True,
                labels=["Outliers", "Clustered"]
            )
            ax.set_title(col)
        plt.suptitle(f"Quantitative variables — outliers vs clustered — {run_label}")
        plt.tight_layout()
        path = os.path.join(out_dir, "outliers_quanti.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

    # ── Categorical admission variables ────────────────────────────────────────
    categ_available = [c for c in ADMISSION_CATEG if c in df_noise.columns]
    for col in categ_available:
        modalities  = sorted(df_work[col].dropna().unique())
        n_mod       = len(modalities)
        x           = np.arange(n_mod)
        width       = 0.35

        noise_pct   = df_noise[col].value_counts(normalize=True).mul(100).reindex(modalities, fill_value=0)
        cluster_pct = df_clustered[col].value_counts(normalize=True).mul(100).reindex(modalities, fill_value=0)

        fig, ax = plt.subplots(figsize=(max(8, n_mod * 1.2), 5))
        ax.bar(x - width/2, noise_pct.values,   width, label="Outliers",  color="tab:red",  alpha=0.8)
        ax.bar(x + width/2, cluster_pct.values, width, label="Clustered", color="tab:blue", alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(modalities, rotation=45, ha="right")
        ax.set_ylabel("% patients")
        ax.set_title(f"{col} — outliers vs clustered — {run_label}")
        ax.legend()
        plt.tight_layout()
        path = os.path.join(out_dir, f"outliers_{col}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

    # ── Chief complaint ────────────────────────────────────────────────────────
    if CHIEF_COMPLAINT_COL in df_noise.columns:
        top10 = df_noise[CHIEF_COMPLAINT_COL].value_counts().head(TOP_N_COMPLAINTS)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(top10.index[::-1], top10.values[::-1],
                color="tab:red", alpha=0.8)
        ax.set_xlabel("Number of patients")
        ax.set_title(f"Top {TOP_N_COMPLAINTS} chief complaints — outliers — {run_label}")
        plt.tight_layout()
        path = os.path.join(out_dir, "outliers_chief_complaint.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

    # ── Complaint category ────────────────────────────────────────────────────────
    if "complaint_category" in df_noise.columns:
        cats        = sorted(df_work["complaint_category"].dropna().unique())
        noise_pct   = df_noise["complaint_category"].value_counts(normalize=True).mul(100).reindex(cats, fill_value=0)
        cluster_pct = df_clustered["complaint_category"].value_counts(normalize=True).mul(100).reindex(cats, fill_value=0)

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        palette   = sns.color_palette("tab20", len(cats))
        axes[0].barh(cats, noise_pct.values,   color=palette, alpha=0.85)
        axes[0].set_title("Outliers")
        axes[0].set_xlabel("% patients")
        axes[1].barh(cats, cluster_pct.values, color=palette, alpha=0.85)
        axes[1].set_title("Clustered")
        axes[1].set_xlabel("% patients")
        plt.suptitle(f"Complaint category — outliers vs clustered — {run_label}")
        plt.tight_layout()
        path = os.path.join(out_dir, "outliers_complaint_category.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

    # ── Measurement rates ──────────────────────────────────────────────────────
    available_measured = {k: v for k, v in MEASURED_STATUS_PAIRS.items()
                          if k in df_noise.columns}
    if available_measured:
        rates_noise   = {
            k.replace("is_", "").replace("_measured", ""): df_noise[k].mean() * 100
            for k in available_measured
        }
        rates_cluster = {
            k.replace("is_", "").replace("_measured", ""): df_clustered[k].mean() * 100
            for k in available_measured
        }
        df_rates = pd.DataFrame({
            "outliers":  rates_noise,
            "clustered": rates_cluster,
        }).round(1)

        fig, ax = plt.subplots(figsize=(max(10, len(df_rates) * 1.2), 5))
        sns.heatmap(
            df_rates.T, annot=True, fmt=".1f", cmap="YlOrRd",
            ax=ax, annot_kws={"size": 9}
        )
        ax.set_title(f"Measurement rates (%) — outliers vs clustered — {run_label}")
        plt.tight_layout()
        path = os.path.join(out_dir, "outliers_measured_rates.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

    # ── Status variables ───────────────────────────────────────────────────────
    status_cols = [
        c for c in df_noise.columns
        if c.endswith("_status")
        and c in df_clustered.columns
    ]

    for col in status_cols:
        modalities = sorted(
            set(df_noise[col].dropna().unique()) |
            set(df_clustered[col].dropna().unique())
        )
        if len(modalities) == 0:
            continue

        n_mod  = len(modalities)
        x      = np.arange(n_mod)
        width  = 0.35

        noise_pct   = (
            df_noise[col].value_counts(normalize=True)
            .mul(100)
            .reindex(modalities, fill_value=0)
        )
        cluster_pct = (
            df_clustered[col].value_counts(normalize=True)
            .mul(100)
            .reindex(modalities, fill_value=0)
        )

        # Color map — not_measured = grey, invalid = orange
        color_map_status = {
            "not_measured": "lightgrey",
            "invalid":      "orange",
            "unknown":      "silver",
        }
        default_palette = sns.color_palette("tab10", n_mod)
        bar_colors = [
            color_map_status.get(m, default_palette[i])
            for i, m in enumerate(modalities)
        ]

        fig, axes = plt.subplots(1, 2, figsize=(max(12, n_mod * 1.5), 5),
                                  sharey=False)

        axes[0].bar(
            x, noise_pct.values,
            color=bar_colors, alpha=0.85, edgecolor="white"
        )
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(modalities, rotation=45, ha="right")
        axes[0].set_ylabel("% outliers")
        axes[0].set_title("Outliers")

        axes[1].bar(
            x, cluster_pct.values,
            color=bar_colors, alpha=0.85, edgecolor="white"
        )
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(modalities, rotation=45, ha="right")
        axes[1].set_ylabel("% clustered")
        axes[1].set_title("Clustered")

        plt.suptitle(
            f"{col} — outliers vs clustered — {run_label}",
            y=1.02
        )
        plt.tight_layout()
        path = os.path.join(out_dir, f"outliers_status_{col}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")


    _plot_outliers_summary_visual(df_noise, df_clustered, run_label, out_dir)


    # ── Export CSV ─────────────────────────────────────────────────────────────
    rows = []
    for group, df_g in [("outliers", df_noise), ("clustered", df_clustered)]:
        row = {
            "group": group,
            "n":     len(df_g),
            "pct":   round(len(df_g) / n_total * 100, 1),
        }

        # Quantitative
        for col in quanti_available:
            if col in df_g.columns:
                row[f"{col}_mean"]   = round(df_g[col].mean(), 1)
                row[f"{col}_median"] = round(df_g[col].median(), 1)

        # Triage
        if "triage" in df_g.columns:
            for level in sorted(df_work["triage"].dropna().unique()):
                row[f"triage_{int(level)}_pct"] = round(
                    (df_g["triage"] == level).mean() * 100, 1
                )

        # Measurement flags
        for k in available_measured:
            var = k.replace("is_", "").replace("_measured", "")
            row[f"{var}_measured_pct"] = round(df_g[k].mean() * 100, 1)

        # Status — % per modality
        for col in status_cols:
            for mod in sorted(df_g[col].dropna().unique()):
                row[f"{col}_{mod}_pct"] = round(
                    (df_g[col] == mod).mean() * 100, 1
                )

        rows.append(row)


    pd.DataFrame(rows).to_csv(
        os.path.join(out_dir, "outliers_external_summary.csv"), index=False
    )
    log.info(f"[{run_label}] Outlier external description complete.")




def _plot_outliers_summary_visual(df_noise, df_clustered, run_label, out_dir):
    """
    Visual summary of outlier profile vs clustered patients.
    Covers: triage, age group, key clinical status variables.
    """
    out_subdir = os.path.join(out_dir, "outliers_profile")
    os.makedirs(out_subdir, exist_ok=True)

    # ── 1. Triage distribution ────────────────────────────────────────────────
    if "triage" in df_noise.columns:
        triage_levels = sorted(df_noise["triage"].dropna().unique())
        noise_pct     = df_noise["triage"].value_counts(normalize=True).mul(100).reindex(triage_levels, fill_value=0)
        cluster_pct   = df_clustered["triage"].value_counts(normalize=True).mul(100).reindex(triage_levels, fill_value=0)

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        colors = sns.color_palette("RdYlGn_r", len(triage_levels))
        bars   = axes[0].bar(
            [f"Triage {int(t)}" for t in triage_levels],
            noise_pct.values, color=colors, alpha=0.85
        )
        for bar, v, t in zip(bars, noise_pct.values, triage_levels):
            n = (df_noise["triage"] == t).sum()
            axes[0].text(
                bar.get_x() + bar.get_width() / 2,
                v + 0.5, f"{v:.1f}%\n(n={n})",
                ha="center", fontsize=9
            )
        axes[0].set_ylabel("% of outliers")
        axes[0].set_title("Triage distribution — outliers only")
        axes[0].set_ylim(0, noise_pct.max() * 1.25)

        x = np.arange(len(triage_levels))
        axes[1].bar(x - 0.2, noise_pct.values,   0.4, label="Outliers",  color="tab:red",  alpha=0.8)
        axes[1].bar(x + 0.2, cluster_pct.values, 0.4, label="Clustered", color="tab:blue", alpha=0.8)
        axes[1].set_xticks(x)
        axes[1].set_xticklabels([f"Triage {int(t)}" for t in triage_levels])
        axes[1].set_ylabel("% patients")
        axes[1].set_title("Triage — outliers vs clustered")
        axes[1].legend()

        plt.suptitle(f"Triage profile — {run_label}")
        plt.tight_layout()
        plt.savefig(os.path.join(out_subdir, "outliers_triage_profile.png"), dpi=150, bbox_inches="tight")
        plt.close()

    # ── 2. Age group ──────────────────────────────────────────────────────────
    if "age_group" in df_noise.columns:
        age_groups  = sorted(df_noise["age_group"].dropna().unique())
        noise_pct   = df_noise["age_group"].value_counts(normalize=True).mul(100).reindex(age_groups, fill_value=0)
        cluster_pct = df_clustered["age_group"].value_counts(normalize=True).mul(100).reindex(age_groups, fill_value=0)

        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(len(age_groups))
        ax.bar(x - 0.2, noise_pct.values,   0.4, label="Outliers",  color="tab:red",  alpha=0.8)
        ax.bar(x + 0.2, cluster_pct.values, 0.4, label="Clustered", color="tab:blue", alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(age_groups, rotation=45, ha="right")
        ax.set_ylabel("% patients")
        ax.set_title(f"Age group — outliers vs clustered — {run_label}")
        ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(out_subdir, "outliers_age_group.png"), dpi=150, bbox_inches="tight")
        plt.close()

    # ── 3. Key clinical status variables ─────────────────────────────────────
    # Only pathological modalities — excludes not_measured, normal, off
    STATUS_TO_PLOT = {
        "bp_status":               ["hypertension", "hypotension"],
        "hr_status":               ["tachycardia", "bradycardia"],
        "temp_status":             ["hyperthermia", "hypothermia"],
        "sat_status":              ["hypoxia", "severe hypoxia"],
        "rr_status":               ["tachypnea", "bradypnea"],
        "gcs_status":              ["moderate_impairment", "severe_impairment"],
        "pain_status":             ["severe_pain", "moderate_pain"],
        "cap_blood_sugar_status":  ["hyperglycemia", "hypoglycemia"],
        "hemocue_status":          ["moderate_anemia", "severe_anemia"],
        "breathalyzer_status":     ["positive"],
        "anisocoria_status":       ["yes"],
        "o2_flow_status":          ["on"],
    }

    labels_plot  = []
    noise_vals   = []
    cluster_vals = []

    for base_col, modalities in STATUS_TO_PLOT.items():
        if base_col not in df_noise.columns:
            continue
        for mod in modalities:
            labels_plot.append(f"{base_col.replace('_status','')}\n{mod}")
            noise_vals.append((df_noise[base_col] == mod).mean() * 100)
            cluster_vals.append((df_clustered[base_col] == mod).mean() * 100)

    if labels_plot:
        x   = np.arange(len(labels_plot))
        fig, ax = plt.subplots(figsize=(max(14, len(labels_plot) * 1.2), 6))
        ax.bar(x - 0.2, noise_vals,   0.4, label="Outliers",  color="tab:red",  alpha=0.8)
        ax.bar(x + 0.2, cluster_vals, 0.4, label="Clustered", color="tab:blue", alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(labels_plot, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("% patients")
        ax.set_title(f"Pathological clinical status — outliers vs clustered — {run_label}")
        ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(out_subdir, "outliers_clinical_status.png"), dpi=150, bbox_inches="tight")
        plt.close()

    log.info(f"Outlier profile plots saved → {out_subdir}")



In [ ]:
# ==============================================================================
# TABLE 1 — Descriptive statistics by cluster
# ==============================================================================

def build_table1(df, cluster_col, out_dir, run_label,
                 quanti_cols=None, categ_cols=None, status_cols=None,
                 internal_cols=None):
    """
    Builds two publication-ready Table 1 CSVs:
    - table1_external.csv : sociodemographic + clinical variables
    - table1_internal.csv : resource utilization variables (clustering features)
    """
    os.makedirs(out_dir, exist_ok=True)

    clusters = [c for c in CLUSTER_ORDER if c in df[cluster_col].unique()]
    n_total = len(df)

    # ── Helper ────────────────────────────────────────────────────────────────
    def fmt_quanti(series):
        m, s = series.mean(), series.std()
        return f"{m:.1f} ± {s:.1f}"

    def fmt_median(series):
        med = series.median()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        return f"{med:.1f} [{q1:.1f}–{q3:.1f}]"

    def fmt_pct(n, total):
        return f"{n} ({100 * n / total:.1f}%)"

    # ── Build table ───────────────────────────────────────────────────────────
    def build_table(var_groups, fname):
        rows = []

        for var_type, var, label in var_groups:
            if var not in df.columns:
                continue

            row = {"Variable": label, "Category": "", "Total": ""}

            if var_type == "quanti_mean":
                row["Total"] = fmt_quanti(df[var].dropna())
                for c in clusters:
                    sub = df[df[cluster_col] == c][var].dropna()
                    row[c] = fmt_quanti(sub)

            elif var_type == "quanti_median":
                row["Total"] = fmt_median(df[var].dropna())
                for c in clusters:
                    sub = df[df[cluster_col] == c][var].dropna()
                    row[c] = fmt_median(sub)

            elif var_type == "categ":
                row["Total"] = ""
                for c in clusters:
                    row[c] = ""
                rows.append(row)

                modalities = sorted(df[var].dropna().unique())
                for mod in modalities:
                    sub_row = {"Variable": "", "Category": str(mod)}
                    n_tot = (df[var] == mod).sum()
                    sub_row["Total"] = fmt_pct(n_tot, n_total)
                    for c in clusters:
                        sub = df[df[cluster_col] == c]
                        n_c = (sub[var] == mod).sum()
                        sub_row[c] = fmt_pct(n_c, len(sub))
                    rows.append(sub_row)
                continue

            elif var_type == "binary":
                n_tot = df[var].sum()
                row["Total"] = fmt_pct(int(n_tot), n_total)
                for c in clusters:
                    sub = df[df[cluster_col] == c]
                    n_c = sub[var].sum()
                    row[c] = fmt_pct(int(n_c), len(sub))

            rows.append(row)

        # Header row with N
        header = {"Variable": "N", "Category": "", "Total": fmt_pct(n_total, n_total)}
        for c in clusters:
            n_c = (df[cluster_col] == c).sum()
            header[c] = fmt_pct(n_c, n_total)

        df_table = pd.DataFrame([header] + rows)
        path = os.path.join(out_dir, fname)
        df_table.to_csv(path, index=False)
        print(f"Saved: {path}")
        return df_table

    # ── TABLE 1 EXTERNAL ─────────────────────────────────────────────────────
    external_vars = [
        # (type, colonne, label affiché)
        ("quanti_mean", "age", "Age (mean ± SD)"),
        ("quanti_median", "age", "Age (median [IQR])"),
        ("categ", "sex", "Sex"),
        ("categ", "age_group", "Age group"),
        ("categ", "transport_grouped", "Transport mode"),
        ("quanti_median", "duration_triage_ioa_min", "Duration triage→IOA, min (median [IQR])"),
        ("categ", "triage", "Triage level"),
        ("categ", "complaint_category", "Chief complaint category"),
        # Status — pathological only
        ("categ", "bp_status", "Blood pressure status"),
        ("categ", "hr_status", "Heart rate status"),
        ("categ", "temp_status", "Temperature status"),
        ("categ", "sat_status", "SpO2 status"),
        ("categ", "pain_status", "Pain status"),
        ("categ", "gcs_status", "GCS status"),
    ]
    build_table(external_vars, f"table1_external_{run_label}.csv")

    # ── TABLE 2 INTERNAL ─────────────────────────────────────────────────────
    internal_vars = [
        ("binary", "has_blood_test", "Blood work"),
        ("binary", "has_culture", "Microbiological cultures"),
        ("binary", "has_lumbar_puncture", "Lumbar puncture"),
        ("binary", "has_blood_gas", "Arterial blood gas"),
        ("binary", "had_ekg", "EKG"),
        ("binary", "has_xray", "X-ray"),
        ("binary", "has_ct_scan", "CT scan"),
        ("binary", "has_ultrasound", "Ultrasound"),
        ("binary", "has_mri", "MRI"),
        ("quanti_median", "bio_exam_count", "Number of biological exams (median [IQR])"),
        ("quanti_median", "imaging_exam_count", "Number of imaging exams (median [IQR])"),
        ("binary", "hospitalization", "Hospitalization"),
        ("binary", "observation_unit", "Observation unit (UHCD)"),
    ]
    build_table(internal_vars, f"table2_internal_{run_label}.csv")




In [65]:
# ==============================================================================
# CALL — CLUSTER DESCRIPTION — FULL DATASET - S2 balanced 5 clusters
# ==============================================================================

run_label = "s2_balanced"
scaler    = "minmax"

FINAL_PARAMS = {
    "s2_balanced_minmax" : (3406, 34),
}

mcs, ms = FINAL_PARAMS[f"{run_label}_{scaler}"]
print(f"mcs={mcs:,} | ms={ms}")

# ── Input CSV ─────────────────────────────────────────────────────────────────
csv_path = os.path.join(
    FULL_OUTPUT_DIR, "With_counts",
    scaler, run_label,
    f"final_mcs{mcs}_ms{ms}",
    f"clustering_{run_label}_{scaler}.csv"
)

# ── Output directory ──────────────────────────────────────────────────────────
out_dir = os.path.join(
    FULL_OUTPUT_DIR, "With_counts",
    scaler, run_label,
    f"final_mcs{mcs}_ms{ms}",
    "description"
)

# ── Load clustering CSV ───────────────────────────────────────────────────────
df_clust = pd.read_csv(csv_path, low_memory=False)
labels   = df_clust["cluster"].values
idx      = df_clust.index

print(f"Loaded: {csv_path}")
print(f"N={len(df_clust):,} | clusters={len(set(labels[labels != -1]))}")

df_clust["cluster_label"] = df_clust["cluster"].map(CLUSTER_LABELS)
df_clust["cluster_label"] = pd.Categorical(
    df_clust["cluster_label"],
    categories=CLUSTER_ORDER,
    ordered=True
)

# ── Cluster description ───────────────────────────────────────────────────────
describe_clusters(
    df             = df_clust,
    labels         = labels,
    idx            = idx,
    run_label      = run_label,
    out_dir        = out_dir,
    cluster_labels = CLUSTER_LABELS_5,
    cluster_order  = CLUSTER_ORDER_5,
)

# ── Outlier description ───────────────────────────────────────────────────────

describe_outliers_external(
    df_clust       = df_clust,
    labels         = labels,
    idx            = idx,
    run_label      = run_label,
    out_dir        = os.path.join(out_dir, "outliers"),
    cluster_labels = CLUSTER_LABELS_5,
    cluster_order  = CLUSTER_ORDER_5,
)

# ── Reverse distribution ──────────────────────────────────────────────────────
df_desc            = df_clust.loc[idx].copy()
df_desc["cluster"] = labels
df_desc            = df_desc[df_desc["cluster"] != -1]

        # ── Remapping ─────────────────────────────────────────────────────────────────
if CLUSTER_LABELS:
    df_desc["cluster"] = df_desc["cluster"].map(CLUSTER_LABELS_5)
    clusters = CLUSTER_ORDER_5
else:
    clusters = sorted(df_desc["cluster"].unique())

_plot_reverse_distribution(
    df_desc   = df_desc,
    clusters  = clusters,
    run_label = f"{run_label}_{scaler}",
    out_dir   = os.path.join(out_dir, "reverse_distribution"),

)


# ── Save new varaible with lbels into CSV ────────────────────
df_out = df_clust.copy()
df_out["cluster_label"] = df_out["cluster"].map(CLUSTER_LABELS_5)
df_out["cluster_label"] = pd.Categorical(
    df_out["cluster_label"],
    categories=CLUSTER_ORDER_5,
    ordered=True
)

output_csv = os.path.join(
    FULL_OUTPUT_DIR, "With_counts",
    scaler, run_label,
    f"final_mcs{mcs}_ms{ms}",
    f"clustering_{run_label}_{scaler}_mcs{mcs}_ms{ms}_labeled.csv"
)
df_out.to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")

# ── CALL table 1──────────────────────────────────────────────────────────────────────
build_table1(
    df=df_clust.copy(),
    cluster_col="cluster_label",
    out_dir=os.path.join(out_dir, "tables"),
    run_label=f"{run_label}_{scaler}",
)

mcs=2,271 | ms=15


INFO | [s2_balanced] Description of 10 clusters (52580 patients)


Loaded: Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs2271_ms15/clustering_s2_balanced_minmax.csv
N=56,781 | clusters=9


INFO | Saved: Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs2271_ms15/description/cluster_sizes.png
/tmp/ipykernel_3283152/3145705777.py:137: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, patch_artist=True, labels=[f"C{c}" for c in clusters])
/tmp/ipykernel_3283152/3145705777.py:137: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, patch_artist=True, labels=[f"C{c}" for c in clusters])
/tmp/ipykernel_3283152/3145705777.py:137: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, patch_artist=True, labels=[f"C{c}" for c in cluster

Saved: Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs2271_ms15/clustering_s2_balanced_minmax_mcs2271_ms15_labeled.csv


In [ ]:
# ==============================================================================
# CALL — CLUSTER DESCRIPTION — FULL DATASET - S2 balanced 9 clusters
# ==============================================================================

run_label = "s2_balanced"
scaler    = "minmax"

FINAL_PARAMS = {
    "s2_balanced_minmax" : (2271, 15),
}

mcs, ms = FINAL_PARAMS[f"{run_label}_{scaler}"]
print(f"mcs={mcs:,} | ms={ms}")

# ── Input CSV ─────────────────────────────────────────────────────────────────
csv_path = os.path.join(
    FULL_OUTPUT_DIR, "With_counts",
    scaler, run_label,
    f"final_mcs{mcs}_ms{ms}",
    f"clustering_{run_label}_{scaler}.csv"
)

# ── Output directory ──────────────────────────────────────────────────────────
out_dir = os.path.join(
    FULL_OUTPUT_DIR, "With_counts",
    scaler, run_label,
    f"final_mcs{mcs}_ms{ms}",
    "description"
)

# ── Load clustering CSV ───────────────────────────────────────────────────────
df_clust = pd.read_csv(csv_path, low_memory=False)
labels   = df_clust["cluster"].values
idx      = df_clust.index

print(f"Loaded: {csv_path}")
print(f"N={len(df_clust):,} | clusters={len(set(labels[labels != -1]))}")

df_clust["cluster_label"] = df_clust["cluster"].map(CLUSTER_LABELS_9)
df_clust["cluster_label"] = pd.Categorical(
    df_clust["cluster_label"],
    categories=CLUSTER_ORDER_9,
    ordered=True
)

# ── Cluster description ───────────────────────────────────────────────────────
describe_clusters(
    df             = df_clust,
    labels         = labels,
    idx            = idx,
    run_label      = run_label,
    out_dir        = out_dir,
    cluster_labels = CLUSTER_LABELS_9,
    cluster_order  = CLUSTER_ORDER_9,
)

# ── Outlier description ───────────────────────────────────────────────────────

describe_outliers_external(
    df_clust       = df_clust,
    labels         = labels,
    idx            = idx,
    run_label      = run_label,
    out_dir        = os.path.join(out_dir, "outliers"),
    cluster_labels = CLUSTER_LABELS_9,
    cluster_order  = CLUSTER_ORDER_9,
)

# ── Reverse distribution ──────────────────────────────────────────────────────
df_desc            = df_clust.loc[idx].copy()
df_desc["cluster"] = labels
df_desc            = df_desc[df_desc["cluster"] != -1]

        # ── Remapping ─────────────────────────────────────────────────────────────────
if CLUSTER_LABELS:
    df_desc["cluster"] = df_desc["cluster"].map(CLUSTER_LABELS_9)
    clusters = CLUSTER_ORDER_9
else:
    clusters = sorted(df_desc["cluster"].unique())

_plot_reverse_distribution(
    df_desc   = df_desc,
    clusters  = clusters,
    run_label = f"{run_label}_{scaler}",
    out_dir   = os.path.join(out_dir, "reverse_distribution"),

)


# ── Save new varaible with lbels into CSV ────────────────────
df_out = df_clust.copy()
df_out["cluster_label"] = df_out["cluster"].map(CLUSTER_LABELS_9)
df_out["cluster_label"] = pd.Categorical(
    df_out["cluster_label"],
    categories=CLUSTER_ORDER_9,
    ordered=True
)

output_csv = os.path.join(
    FULL_OUTPUT_DIR, "With_counts",
    scaler, run_label,
    f"final_mcs{mcs}_ms{ms}",
    f"clustering_{run_label}_{scaler}_mcs{mcs}_ms{ms}_labeled.csv"
)
df_out.to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")

# ── CALL ──────────────────────────────────────────────────────────────────────
build_table1(
    df=df_clust.copy(),
    cluster_col="cluster_label",
    out_dir=os.path.join(out_dir, "tables"),
    run_label=f"{run_label}_{scaler}",
)

Triage
Les outliers sont majoritairement triage 3 (49.9%) et triage 2 (26.7%) — très peu de triage 4-5 (23.4% combinés). C'est l'inverse des patients clustérisés où triage 4 est bien plus représenté (28.6%). Les outliers sont donc des patients modérément à sévèrement urgents — pas des passages simples.
Age
Age moyen et médian quasi identiques entre outliers (47.4 / 43.0) et clustérisés (47.5 / 43.0) — l'âge ne distingue pas les outliers.
Hypertension
28.2% des outliers sont hypertendus vs 26.0% des clustérisés — légèrement plus mais pas frappant.
Tachycardie
10.5% des outliers vs 8.5% des clustérisés — légèrement plus de tachycardie chez les outliers.
Hypoxie
2.0% + 0.2% (hypoxie + hypoxie sévère) chez outliers vs 1.9% + 0.2% chez clustérisés — identique.
O2 flow on
3.4% outliers vs 4.0% clustérisés — légèrement moins d'O2 chez les outliers.
Douleur sévère
0.7% outliers vs 0.6% clustérisés — identique.
Conclusion interprétative
Les outliers ne se distinguent pas par leurs caractéristiques cliniques à l'admission — même âge, même profil de constantes, même distribution des statuts pathologiques. Ce qui les distingue c'est uniquement leur profil de consommation de ressources mixte (observation + biologie intensive sans hospitalisation) comme tu l'avais vu dans la heatmap. C'est cohérent avec l'idée que ce sont des patients en incertitude décisionnelle — cliniquement similaires aux autres mais avec un parcours atypique difficile à clustériser.